## Intro

In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/xiluvamabasa/german-credit-risk-pred/german_credit_data (2).csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/xiluvamabasa/german-credit-risk-pred/german_credit_data (2).csv')

df.head()

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,2,49,male,1,own,little,NaN,2096,12,education,good
3,3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,4,53,male,2,free,little,little,4870,24,car,bad


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1000 non-null   int64 
 1   Age               1000 non-null   int64 
 2   Sex               1000 non-null   object
 3   Job               1000 non-null   int64 
 4   Housing           1000 non-null   object
 5   Saving accounts   817 non-null    object
 6   Checking account  606 non-null    object
 7   Credit amount     1000 non-null   int64 
 8   Duration          1000 non-null   int64 
 9   Purpose           1000 non-null   object
 10  Risk              1000 non-null   object
dtypes: int64(5), object(6)
memory usage: 86.1+ KB


#### Drop unnecessary columns

In [4]:
df.drop(columns=['Unnamed: 0'], inplace=True)

#### Clean column names

In [5]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

#### Handle missing values

In [6]:
df['saving_accounts'] = df['saving_accounts'].fillna('unknown')
df['checking_account'] = df['checking_account'].fillna('unknown')

In [7]:
df['risk'].value_counts()

risk
good    700
bad     300
Name: count, dtype: int64

#### Encode risk column


In [8]:
df['risk'] = df['risk'].map({'good': 0, 'bad': 1})

In [9]:
df.describe()

,age,job,credit_amount,duration,risk
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,35.546000,1.904000,3271.258000,20.903000,0.300000
std,11.375469,0.653614,2822.736876,12.058814,0.458487
min,19.000000,0.000000,250.000000,4.000000,0.000000
25%,27.000000,2.000000,1365.500000,12.000000,0.000000
50%,33.000000,2.000000,2319.500000,18.000000,0.000000
75%,42.000000,2.000000,3972.250000,24.000000,1.000000
max,75.000000,3.000000,18424.000000,72.000000,1.000000


In [10]:
df.head()

,age,sex,job,housing,saving_accounts,checking_account,credit_amount,duration,purpose,risk
0,67,male,2,own,unknown,little,1169,6,radio/TV,0
1,22,female,2,own,little,moderate,5951,48,radio/TV,1
2,49,male,1,own,little,unknown,2096,12,education,0
3,45,male,2,free,little,little,7882,42,furniture/equipment,0
4,53,male,2,free,little,little,4870,24,car,1


### Feature Engineering

In [11]:
# Loan intensity
df['credit_per_month'] = df['credit_amount'] / df['duration']

In [12]:
# Age Group
df['age_group'] = pd.cut(df['age'],
                         bins =[18, 25, 35, 50, 65, 100],
                         labels=['18-25', '26-35', '36-50', '51-65', '66+'])

In [13]:
# Credit Duration Category
df['duration_group'] = pd.cut(df['duration'],
                             bins=[0,12,24,48,100],
                             labels=['short', 'medium', 'long', 'very_long'])

In [14]:
# High Credit Flag
df['high_credit'] = (df['credit_amount'] > df['credit_amount'].median()).astype(int)

In [15]:
df.groupby('risk')['credit_amount'].mean()

risk
0    2985.457143
1    3938.126667
Name: credit_amount, dtype: float64

In [20]:
df.groupby('risk')['duration'].mean()

risk
0    19.207143
1    24.860000
Name: duration, dtype: float64

In [16]:
df.groupby('risk')['credit_per_month'].mean()

risk
0    165.819730
1    172.044031
Name: credit_per_month, dtype: float64

In [17]:
df.to_csv('cleaned_credit_data.csv', index=False)

In [18]:
import sqlite3

df = pd.read_csv('cleaned_credit_data.csv')
conn = sqlite3.connect('credit_risk.db')
df.to_sql('credit_data', conn, if_exists ='replace', index=False)

1000

In [19]:
pd.read_sql("SELECT * FROM credit_data LIMIT 5", conn)

,age,sex,job,housing,saving_accounts,checking_account,credit_amount,duration,purpose,risk,credit_per_month,age_group,duration_group,high_credit
0,67,male,2,own,unknown,little,1169,6,radio/TV,0,194.833333,66+,short,0
1,22,female,2,own,little,moderate,5951,48,radio/TV,1,123.979167,18-25,long,1
2,49,male,1,own,little,unknown,2096,12,education,0,174.666667,36-50,short,0
3,45,male,2,free,little,little,7882,42,furniture/equipment,0,187.666667,36-50,long,1
4,53,male,2,free,little,little,4870,24,car,1,202.916667,51-65,medium,1


In [20]:
# 1. Default/Risk Distribution

query = """ 
SELECT risk, COUNT(*) AS total_customers
FROM credit_data
GROUP BY risk
"""
pd.read_sql(query, conn)

,risk,total_customers
0,0,700
1,1,300


In [21]:
# 2. Average Credit by Risk

query = """
SELECT risk, AVG(credit_amount) AS avg_credit
FROM credit_data
GROUP BY risk
"""
pd.read_sql(query, conn)

,risk,avg_credit
0,0,2985.457143
1,1,3938.126667


In [22]:
# 3. Risk vs Loan durarion

query = """
SELECT duration_group, AVG(risk) AS risk_rate
FROM credit_data
GROUP BY duration_group
ORDER BY risk_rate DESC
"""
pd.read_sql(query, conn)

,duration_group,risk_rate
0,very_long,0.500000
1,long,0.439252
2,medium,0.296837
3,short,0.211699


In [23]:
# 4. Risk by Housing Tyope

query = """
SELECT housing, AVG(risk) AS risk_rate
FROM credit_data
GROUP BY housing
ORDER BY risk_rate DESC
"""
pd.read_sql(query, conn)

,housing,risk_rate
0,free,0.407407
1,rent,0.391061
2,own,0.260870


In [24]:
# 5. Credit intensity vs Risk

query = """
SELECT
    CASE
        WHEN credit_per_month > (SELECT AVG(credit_per_month) FROM credit_data)
        THEN 'High Load'
        ELSE 'Normal Load'
    END AS credit_burden,
    AVG(risk) AS risk_rate
FROM credit_data
GROUP BY credit_burden
"""
pd.read_sql(query, conn)

,credit_burden,risk_rate
0,High Load,0.263610
1,Normal Load,0.319508


#### SQL Insights
- The dataset shows a class imbalance, with approximately 70% low-rsik customers and 30% risk customers. This reflects real-worl lending data where defaults are less frequent but financially significant
- Customers classified as high risk tend to have higer average credit amounts, suggesting that larger loan sizes may be associated with increased default risk.
- Longer loan durations are associated with higher default risk, indicating that extended repayment periods may increase exposure to borrower instability or repaymnt failure.
- Customers living in free housing exhibit higher risk rates compared to homeowners, suggesting that housing stability may be a strong proxy for financial stability.
- Surprisingly, customers classified under normal credit burden showed higher risk rates than those under high credit burden. This may indicate that credit burden is not linear in its relationship with risjm or that additional behavioral factors influence default probability.

### Machine Learning modeling

In [25]:
# target
y = df['risk']

# Features
X = df.drop(columns=['risk'])

#### Handle cateforical viaribales

In [26]:
X = pd.get_dummies(X, drop_first=True)

#### Train-test-split

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Model 1 - Logistic Regresion

In [36]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

#### Scale data

In [39]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [40]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [41]:
y_pred_lr = lr.predict(X_test_scaled)

#### Evaluation metrics for Logistic Regression

In [43]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[130  11]
 [ 33  26]]
              precision    recall  f1-score   support

           0       0.80      0.92      0.86       141
           1       0.70      0.44      0.54        59

    accuracy                           0.78       200
   macro avg       0.75      0.68      0.70       200
weighted avg       0.77      0.78      0.76       200



#### Model 2 - Random Forest Classifier

In [45]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [46]:
y_pred_rf = rf.predict(X_test)

#### Evaluation metrics for Random Forest Classifier

In [47]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[130  11]
 [ 36  23]]
              precision    recall  f1-score   support

           0       0.78      0.92      0.85       141
           1       0.68      0.39      0.49        59

    accuracy                           0.77       200
   macro avg       0.73      0.66      0.67       200
weighted avg       0.75      0.77      0.74       200



#### Models comparison
- Logistic Regression outperformed Random Forest in identifying high-risk customers, achieving a higher recall score (0.44 vs 0.39). This suggests it is more effective at detecting potential defaulters, which is critical in financial risk management.
- However, both models show relatively low recall, indicating that a significant portion of high-risk customers are still being misclassified. This suggests room for improvement through feature engineering or model tuning.

- The dataset likely has linear relationships hence Logistic Regression handled it better than Random Forest

### Apply class weighting to Logistic Regression

In [48]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [49]:
y_pred_lr = lr.predict(X_test_scaled)

In [50]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[99 42]
 [22 37]]
              precision    recall  f1-score   support

           0       0.82      0.70      0.76       141
           1       0.47      0.63      0.54        59

    accuracy                           0.68       200
   macro avg       0.64      0.66      0.65       200
weighted avg       0.71      0.68      0.69       200



#### Insights
- After applying class weighting, the Logistic Regression model showed a significant improvement in recall for hogh-rsik customers (from 44% to 63%), indicating a better ability to detect potential defaulters.
- Although overall accuracy decreased, this trade-off is acceptable in a financial risk context where identifyinh high-rsik individuals is more critical than overall prediction accuracy.

### Feature Importance

In [51]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr.coef_[0]
})
feature_importance = feature_importance.sort_values(by='coefficient', ascending=False)
feature_importance.head(10)

,feature,coefficient
3,duration,0.759400
4,credit_per_month,0.281169
27,duration_group_medium,0.148544
28,duration_group_short,0.147122
1,job,0.136934
2,credit_amount,0.090145
18,purpose_education,0.060133
21,purpose_repairs,0.020012
9,saving_accounts_moderate,0.013524
17,purpose_domestic appliances,-0.039836


In [52]:
rf_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})
rf_importance = rf_importance.sort_values(by='importance', ascending=False)
rf_importance.head(10)

,feature,importance
4,credit_per_month,0.157407
2,credit_amount,0.154343
0,age,0.126753
3,duration,0.101178
15,checking_account_unknown,0.070802
1,job,0.043141
16,purpose_car,0.026149
12,saving_accounts_unknown,0.024895
6,sex_male,0.024445
13,checking_account_moderate,0.023833


### Insights
- Feature importance analysis reveals that both raw financial variables and engineered feautures play a cricital role in predicting credit risk. In particular, repayment burden and loan duration emerged as the strongest predictors, highlighting the importance of affordability and loan structuring in risk assessment.

## Conclusion